In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
from pathlib import Path
import os
import subprocess
import sys

CODE_ROOT = Path("/kaggle/input/datasets/ananthakrishnan86/gdl-codebase")
INPUT_ROOT = Path("/kaggle/input/datasets")
SPLITS_PATH = CODE_ROOT / "benchmarks/airfrans/kaggle_splits_96_16_40.json"
RUN_ROOT = Path("/kaggle/working/airfrans_local_30epoch")

def has_airfrans_cases(root):
    if not root.is_dir():
        return False

    for case_dir in root.iterdir():
        if (
            case_dir.is_dir()
            and case_dir.name.startswith("airFoil2D_SST_")
            and (case_dir / f"{case_dir.name}_internal.vtu").is_file()
            and (case_dir / f"{case_dir.name}_aerofoil.vtp").is_file()
        ):
            return True

    return False

data_roots = []
for owner_dir in INPUT_ROOT.iterdir():
    if not owner_dir.is_dir():
        continue

    for dataset_dir in owner_dir.iterdir():
        if not dataset_dir.is_dir() or dataset_dir == CODE_ROOT:
            continue

        for candidate in (dataset_dir, dataset_dir / "Dataset"):
            if has_airfrans_cases(candidate):
                data_roots.append(candidate)

if len(data_roots) != 1:
    raise RuntimeError(f"Expected one preprocessed AirfRANS root, found: {data_roots}")

DATA_ROOT = data_roots[0]

assert (CODE_ROOT / "benchmarks/airfrans/train.py").is_file()
assert SPLITS_PATH.is_file(), f"Missing split file: {SPLITS_PATH}"

import shutil

if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
assert not RUN_ROOT.exists(), f"Output directory already exists: {RUN_ROOT}"

env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(CODE_ROOT / "benchmarks/airfrans/requirements-kaggle.txt"),
    ],
    check=True,
    env=env,
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")

print(f"Dataset: {DATA_ROOT}")
print(f"Splits:  {SPLITS_PATH}")
print(f"GPU:     {torch.cuda.get_device_name(0)}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "benchmarks.airfrans.train",
        "--data",
        str(DATA_ROOT),
        "--splits",
        str(SPLITS_PATH),
        "--output",
        str(RUN_ROOT),
        "--models",
        "local",
        "--epochs",
        "30",
        "--points",
        "32000",
        "--validation-interval",
        "5",
        "--seed",
        "2026",
    ],
    check=True,
    env=env,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "benchmarks.airfrans.evaluate",
        "--data",
        str(DATA_ROOT),
        "--splits",
        str(SPLITS_PATH),
        "--run-dir",
        str(RUN_ROOT),
    ],
    check=True,
    env=env,
)

print(f"Training and evaluation complete: {RUN_ROOT}")

Dataset: /kaggle/input/datasets/salillokhande8/airfrans-dataset-zip/Dataset
Splits:  /kaggle/input/datasets/ananthakrishnan86/gdl-codebase/benchmarks/airfrans/kaggle_splits_96_16_40.json
GPU:     Tesla T4


KeyboardInterrupt: 

In [2]:
from pathlib import Path
import os
import subprocess
import sys

CODE_ROOT = Path("/kaggle/input/datasets/ananthakrishnan86/gdl-codebase")
INPUT_ROOT = Path("/kaggle/input/datasets")
SPLITS_PATH = CODE_ROOT / "benchmarks/airfrans/kaggle_splits_96_16_40.json"
RUN_ROOT = Path("/kaggle/working/airfrans_gnn_latentattention32_30epoch")

env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
def has_airfrans_cases(root):
    if not root.is_dir():
        return False

    for case_dir in root.iterdir():
        if (
            case_dir.is_dir()
            and case_dir.name.startswith("airFoil2D_SST_")
            and (case_dir / f"{case_dir.name}_internal.vtu").is_file()
            and (case_dir / f"{case_dir.name}_aerofoil.vtp").is_file()
        ):
            return True

    return False
    
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(CODE_ROOT / "benchmarks/airfrans/requirements-kaggle.txt"),
    ],
    check=True,
    env=env,
)
data_roots = []
for owner_dir in INPUT_ROOT.iterdir():
    if not owner_dir.is_dir():
        continue

    for dataset_dir in owner_dir.iterdir():
        if not dataset_dir.is_dir() or dataset_dir == CODE_ROOT:
            continue

        for candidate in (dataset_dir, dataset_dir / "Dataset"):
            if has_airfrans_cases(candidate):
                data_roots.append(candidate)

if len(data_roots) != 1:
    raise RuntimeError(f"Expected one preprocessed AirfRANS root, found: {data_roots}")

DATA_ROOT = data_roots[0]
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator.")
import shutil

if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
assert not RUN_ROOT.exists(), f"Output directory already exists: {RUN_ROOT}"


commands = [
    [
        sys.executable, "-m", "benchmarks.airfrans.train",
        "--data", str(DATA_ROOT),
        "--splits", str(SPLITS_PATH),
        "--output", str(RUN_ROOT),
        "--models", "gnn",
        "--epochs", "398",
        "--points", "32000",
        "--validation-interval", "20",
        "--seed", "2026",
    ],
    [
        sys.executable, "-m", "benchmarks.airfrans.evaluate",
        "--data", str(DATA_ROOT),
        "--splits", str(SPLITS_PATH),
        "--run-dir", str(RUN_ROOT),
    ],
]

for command in commands:
    subprocess.run(command, check=True, env=env)

print(f"Training and evaluation complete: {RUN_ROOT}")

Loaded 96 training and 16 validation cases in 579.3s
gnn {'epoch': 1, 'training_loss': 2.1700144351149597, 'validation_loss': 0.966863552108407, 'elapsed_seconds': 16.133735163999972}
gnn {'epoch': 2, 'training_loss': 1.8457145099528134, 'elapsed_seconds': 30.061813153999992}
gnn {'epoch': 3, 'training_loss': 1.6751065809900563, 'elapsed_seconds': 43.992613776999974}
gnn {'epoch': 4, 'training_loss': 1.606548627295221, 'elapsed_seconds': 57.93254230100001}
gnn {'epoch': 5, 'training_loss': 1.6062338473275304, 'elapsed_seconds': 71.87337057499997}
gnn {'epoch': 6, 'training_loss': 1.48421834083274, 'elapsed_seconds': 85.79706024300003}
gnn {'epoch': 7, 'training_loss': 1.4866307124805946, 'elapsed_seconds': 99.74654830300005}
gnn {'epoch': 8, 'training_loss': 1.5413845131018509, 'elapsed_seconds': 113.77011309700003}
gnn {'epoch': 9, 'training_loss': 1.4035048144093405, 'elapsed_seconds': 127.77770764900004}
gnn {'epoch': 10, 'training_loss': 1.4132323996940006, 'elapsed_seconds': 141.8

In [3]:
from pathlib import Path
import shutil
from IPython.display import FileLink, display

run_dir = Path("/kaggle/working/airfrans_gnn_latentattention32_30epoch")
assert (run_dir / "gnn.pt").is_file(), "Checkpoint not found"
assert (run_dir / "evaluation/metrics.json").is_file(), "Evaluation output not found"

archive = shutil.make_archive(
    str(run_dir),
    "zip",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)
display(FileLink(archive))

/kaggle/working/airfrans_gnn_latentattention32_30epoch.zip

In [ ]:
# Area-weighted latent-attention GNN variants; same split, seed, points and epochs as the 398-epoch "gnn" run.
from pathlib import Path
import os
import shutil
import subprocess
import sys

import torch

CODE_ROOT = Path("/kaggle/input/datasets/ananthakrishnan86/gdl-codebase")
INPUT_ROOT = Path("/kaggle/input/datasets")
SPLITS_PATH = CODE_ROOT / "benchmarks/airfrans/kaggle_splits_96_16_40.json"
MODELS = ["gnn_area_read", "gnn_area"]  # run one per session if 2x398 epochs exceeds the time limit
RUN_ROOT = Path("/kaggle/working/airfrans_gnn_area_398epoch")

assert (CODE_ROOT / "model.py").is_file(), "Upload the updated code dataset (needs model.py)."
assert "area_heads" in (CODE_ROOT / "model.py").read_text(), "Code dataset is outdated."
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator.")

env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(CODE_ROOT / "benchmarks/airfrans/requirements-kaggle.txt")],
    check=True,
    env=env,
)

def has_airfrans_cases(root):
    return root.is_dir() and any(
        case.is_dir()
        and case.name.startswith("airFoil2D_SST_")
        and (case / f"{case.name}_internal.vtu").is_file()
        and (case / f"{case.name}_aerofoil.vtp").is_file()
        for case in root.iterdir()
    )

data_roots = [
    candidate
    for owner in INPUT_ROOT.iterdir() if owner.is_dir()
    for dataset in owner.iterdir() if dataset.is_dir() and dataset != CODE_ROOT
    for candidate in (dataset, dataset / "Dataset") if has_airfrans_cases(candidate)
]
if len(data_roots) != 1:
    raise RuntimeError(f"Expected one preprocessed AirfRANS root, found: {data_roots}")
DATA_ROOT = data_roots[0]

if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)

for command in (
    [sys.executable, "-m", "benchmarks.airfrans.train",
     "--data", str(DATA_ROOT), "--splits", str(SPLITS_PATH), "--output", str(RUN_ROOT),
     "--models", *MODELS, "--epochs", "398", "--points", "32000",
     "--validation-interval", "20", "--seed", "2026"],
    [sys.executable, "-m", "benchmarks.airfrans.evaluate",
     "--data", str(DATA_ROOT), "--splits", str(SPLITS_PATH), "--run-dir", str(RUN_ROOT)],
):
    subprocess.run(command, check=True, env=env)

archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print(f"Training and evaluation complete: {RUN_ROOT}\nDownload: {archive}")